In [0]:
algorithm_name="logistic_regression"
# Set Catalog and Schema for Data Storage and Manipulation 
CATALOG = "test_analytics"
SCHEMA = "raw"
directory_name="lr"
batch_id="70a74319-2c68-4c89-970c-272f4b34c90e"
total_customers=10000

In [0]:
%run ./churn_pipeline


In [0]:
%run ./libs/smart_analytics_agent

In [0]:
%sql
USE CATALOG test_analytics;
USE SCHEMA raw;

In [0]:
%sql
TRUNCATE TABLE churn_predictions_raw;
TRUNCATE TABLE gold.model_metrics_summary;


/Workspace/MOMO-BIGDG/MLOps_NBs/Chstomer_Churn_predictor


📁 ml_pipeline/
├── 📄 data_generator.py       # Generate dummy data
├── 📄 model_trainer.py        # Train ML models (reusable for any algorithm)
├── 📄 model_evaluator.py      # Calculate metrics (algorithm-agnostic)
├── 📄 visualizer.py           # Generate charts (standalone)
├── 📄 mlflow_logger.py        # MLflow logging utilities
├── 📄 report_generator.py     # Export results to different formats
└── 📄 main_pipeline.py        # Orchestrate everything

![](./Folder_Structure.jpg)


Customer Churn Analytics Execution 

In [0]:
if algorithm_name=="logistic_regression":
    !rm ./model_outputs/lr/*
    !rm ./model_outputs/lr/results/*
elif  algorithm_name=="random_forest":
    !rm ./model_outputs/rf/*   
elif  algorithm_name=="gbt":
    !rm ./model_outputs/rf/*       



rm: cannot remove './model_outputs/lr/results': Is a directory


In [0]:
# ========================================================================
# Example 1: Run with Provided Algorithm
# ========================================================================
import os
output_path="./model_outputs/"+directory_name
print(output_path)
os.makedirs(output_path , exist_ok=True) 
pipeline_lr = None
if algorithm_name=="logistic_regression":
    print("\n" + "="*80)
    print("EXAMPLE 1: LOGISTIC REGRESSION")
    print("="*80)
    pipeline_lr = ChurnPredictionPipeline(
    catalog=CATALOG,
    schema=SCHEMA,
    #makedirs("./model_outouts", exist_ok=True)
    viz_path=output_path,
    algorithm=algorithm_name,
    max_iter=100,
    reg_param=0.1
)
elif  algorithm_name=="random_forest":
    print("\n" + "="*80)
    print("EXAMPLE 1: RANDOM FOREST")
    print("="*80)
    pipeline_lr = ChurnPredictionPipeline(
    catalog=CATALOG,
    schema=SCHEMA,
    viz_path=output_path,
    algorithm=algorithm_name,
    num_trees=10,
    max_depth=5,
    min_instances_per_node=20  # Larger leaf size
)
elif  algorithm_name=="gbt":
    print("EXAMPLE 3: GRADIENT BOOSTED TREES")
    print("="*80)
    pipeline_lr = ChurnPredictionPipeline(
        catalog=CATALOG,
        schema=SCHEMA,
        viz_path=output_path,
        algorithm=algorithm_name,
        max_iter=50,
        max_depth=3,
        step_size=0.1
    )




./model_outputs/lr

EXAMPLE 1: LOGISTIC REGRESSION


In [0]:
# ========================================================================
# Example 1: Run with Logistic Regression
# ========================================================================
# print("\n" + "="*80)
# print("EXAMPLE 1: RANDOM FOREST")
# print("="*80)
# import os
# output_path="./model_outputs/rf"
# pipeline_lr = ChurnPredictionPipeline(
#     catalog=CATALOG,
#     schema=SCHEMA,
#     viz_path=output_path,
#     algorithm='random_forest',
#     num_trees=10,
#     max_depth=5,
#     min_instances_per_node=20  # Larger leaf size
# )

In [0]:
# import os 
# # ========================================================================
# # Example 3: Run with Gradient Boosted Trees
# # ========================================================================
# output_path="./model_outputs/gboost"
# os.makedirs(output_path , exist_ok=True) 
# print("\n" + "="*80)
# print("EXAMPLE 3: GRADIENT BOOSTED TREES")
# print("="*80)

# pipeline_lr = ChurnPredictionPipeline(
#     catalog=CATALOG,
#     schema=SCHEMA,
#     viz_path=output_path,
#     algorithm='gbt',
#     max_iter=50,
#     max_depth=3,
#     step_size=0.1
# )

In [0]:
"""
Run the complete pipeline

Args:
    n_customers: Number of customers to generate
    test_size: Test set proportion
    generate_viz: Generate visualizations
    save_viz: Save visualizations to disk
    log_to_mlflow: Log to MLflow
    export_results: Export to multiple formats
"""

print("\n" + "="*80)
print("🚀 STARTING CHURN PREDICTION PIPELINE")
print("="*80)

# ====================================================================
# STEP 1: Generate Data
# ====================================================================
print("\n📊 STEP 1: Generating Data...")
pipeline_lr.data_generator = ChurnDataGenerator(
    pipeline_lr.spark, pipeline_lr.catalog, pipeline_lr.schema, 
    total_customers 
)
df = pipeline_lr.data_generator.create_feature_table()
pipeline_lr.data_generator.create_actuals_table()


🚀 STARTING CHURN PREDICTION PIPELINE

📊 STEP 1: Generating Data...
✓ Created customer_features_ml table with 10000 customers
✓ Created customer_actuals table


In [0]:

# ====================================================================
# STEP 2: Train Model
# ====================================================================
print("\n🤖 STEP 2: Training Model...")
pipeline_lr.trainer = MLModelTrainer(
    feature_cols=pipeline_lr.feature_cols,
    test_size=0.2
)
pipeline_lr.trainer.prepare_data(df)
model = pipeline_lr.trainer.train(algorithm=pipeline_lr.algorithm, **pipeline_lr.algo_params)
predictions = pipeline_lr.trainer.predict()
feature_importance = pipeline_lr.trainer.get_feature_importance()


🤖 STEP 2: Training Model...
✓ Data split: 7950 train, 2050 test

Training logistic_regression model...
✓ Model trained successfully
Making predictions...
✓ Predictions complete


In [0]:
display(predictions.count())

2050

In [0]:
# ====================================================================
# STEP 3: Evaluate Model
# ====================================================================
print("\n📈 STEP 3: Evaluating Model...")
pipeline_lr.evaluator = ModelEvaluator(predictions)
metrics = pipeline_lr.evaluator.calculate_all_metrics()
pipeline_lr.evaluator.print_summary()


📈 STEP 3: Evaluating Model...
✓ All metrics calculated

MODEL PERFORMANCE METRICS
AUC-ROC:          0.9999
AUC-PR:           1.0000
Accuracy:         0.9166
Precision:        0.8991
Recall:           1.0000
F1-Score:         0.9468
Specificity:      0.6755

Confusion Matrix:
True Negatives:   356
False Positives:  171
False Negatives:  0
True Positives:   1523


In [0]:

# ====================================================================
# STEP 4: Generate Visualizations
# ====================================================================
generate_viz=True
viz_paths = "./model_outputs/"+algorithm_name
if generate_viz:
    print("\n🎨 STEP 4: Generating Visualizations...")
    print("\nOutput directory...", pipeline_lr.viz_path)
    pipeline_lr.visualizer = ModelVisualizer(
        pipeline_lr.evaluator, 
        pipeline_lr.feature_cols,
        pipeline_lr.viz_path
    )
    viz_paths = pipeline_lr.visualizer.generate_all_visualizations(
        feature_importance_df=feature_importance,
        save=True,
        show=False
    )


🎨 STEP 4: Generating Visualizations...

Output directory... ./model_outputs/lr

Generating all visualizations...
✓ All visualizations generated


In [0]:

# ====================================================================
# STEP 4: Generate Visualizations
# ====================================================================
# generate_viz=True
# viz_paths = "./model_outputs/lr"
# if generate_viz:
#     print("\n🎨 STEP 4: Generating Visualizations...")
#     print("\nOutput directory...", pipeline_lr.viz_path)
#     pipeline_lr.visualizer = ModelVisualizer(
#         pipeline_lr.evaluator, 
#         pipeline_lr.feature_cols,
#         pipeline_lr.viz_path
#     )
#     viz_paths = pipeline_lr.visualizer.generate_all_visualizations(
#         feature_importance_df=feature_importance,
#         save=True,
#         show=False
#     )
# ====================================================================
# STEP 4: Generate Visualizations
# ====================================================================
# generate_viz=True
# viz_paths = "./model_outputs/rf"
# if generate_viz:
#     print("\n🎨 STEP 4: Generating Visualizations...")
#     print("\nOutput directory...", pipeline_lr.viz_path)
#     pipeline_lr.visualizer = ModelVisualizer(
#         pipeline_lr.evaluator, 
#         pipeline_lr.feature_cols,
#         pipeline_lr.viz_path
#     )
#     viz_paths = pipeline_lr.visualizer.generate_all_visualizations(
#         feature_importance_df=feature_importance,
#         save=True,
#         show=False
#     )

In [0]:
# ====================================================================
# STEP 5: Log to MLflow
# ====================================================================
import os

# Set the MLflow DFS temp directory to a UC volume path
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/test_analytics/raw/ml_models/mlflow_tmp"
pipeline_run_id=-1
log_to_mlflow=True
if log_to_mlflow:
    print("\n📝 STEP 5: Logging to MLflow...")
    experiment_name="/Users/pucitgurru2006@gmail.com/churn_expirement"
    pipeline_lr.mlflow_logger = MLflowLogger(experiment_name)
    
    with pipeline_lr.mlflow_logger.start_run(f"{pipeline_lr.algorithm}_pipeline"):
        # Log parameters
        params = {
            'algorithm': pipeline_lr.algorithm,
            'n_customers': total_customers,
            'test_size': 0.2,
            **pipeline_lr.algo_params
        }
        pipeline_lr.mlflow_logger.log_params(params)
        
        # Log metrics
        pipeline_lr.mlflow_logger.log_metrics(metrics)
        
        # Log artifacts
        if viz_paths:
            pipeline_lr.mlflow_logger.log_artifacts(viz_paths)
        
        # Log model
        pipeline_lr.mlflow_logger.log_model(
            model, 
            pipeline_lr.trainer.train_df,
            predictions,
            pipeline_lr.feature_cols,
            registered_model_name=f"customer_churn_{pipeline_lr.algorithm}"
        )
        
        pipeline_run_id = pipeline_lr.mlflow_logger.run_id
    
    pipeline_lr.mlflow_logger.end_run()
    print("Pipeline Run ID ",pipeline_run_id)


📝 STEP 5: Logging to MLflow...
✓ MLflow run started: logistic_regression_pipeline (ID: 307c3839b41540e7a12f36c8e112b337)
✓ Logged 5 parameters
✓ Logged 11 metrics
✓ Logged 7 artifacts


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/26 03:52:10 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==

✓ Model logged to registry: customer_churn_logistic_regression
✓ MLflow run ended: 307c3839b41540e7a12f36c8e112b337
Pipeline Run ID  307c3839b41540e7a12f36c8e112b337


Created version '7' of model 'test_analytics.raw.customer_churn_logistic_regression'.


In [0]:
        # ====================================================================
        # STEP 6: Export Results
        # ====================================================================
        import os 
        export_results=True
        results= "./model_outputs/"+directory_name+"/results"
        os.makedirs(results , exist_ok=True) 

        if export_results:
            print("\n💾 STEP 6: Exporting Results...")
            pipeline_lr.report_generator = ReportGenerator(pipeline_lr.evaluator, pipeline_lr.trainer,results)
            export_paths = pipeline_lr.report_generator.export_all(visualization_paths=viz_paths)
            
            # Generate GenAI prompt
            genai_prompt, prompt_path = pipeline_lr.report_generator.generate_genai_prompt()
            print("\n" + "="*80)
            print("📤 GenAI PROMPT FOR RESULTS EXPLANATION")
            print("="*80)
            print(genai_prompt)


💾 STEP 6: Exporting Results...

EXPORTING RESULTS TO MULTIPLE FORMATS
✓ Predictions CSV saved: ./model_outputs/lr/results/predictions_20251026_035223.csv
✓ Metrics CSV saved: ./model_outputs/lr/results/metrics_20251026_035223.csv
✓ Threshold analysis CSV saved: ./model_outputs/lr/results/threshold_analysis_20251026_035223.csv
✓ HTML report saved: ./model_outputs/lr/results/model_report_20251026_035223.html
✓ GenAI prompt saved: ./model_outputs/lr/results/genai_prompt_20251026_035223.txt
✓ ALL EXPORTS COMPLETE
✓ GenAI prompt saved: ./model_outputs/lr/results/genai_prompt_20251026_035223.txt

📤 GenAI PROMPT FOR RESULTS EXPLANATION

Analyze the following machine learning model results and provide a comprehensive explanation:

**Model Information:**
- Algorithm: logistic_regression
- Features: recency, frequency, monetary, tenure_days
- Test Set Size: 2050 customers

**Performance Metrics:**
- AUC-ROC: 0.9999
- Accuracy: 0.9166
- Precision: 0.8991
- Recall: 1.0000
- F1-Score: 0.9468
- Spe

In [0]:
from pyspark.sql.functions import lit
TARGET_TABLE = "test_analytics.raw.churn_predictions_tmp"
predictions_with_run_id = predictions.withColumn("pipeline_run_id", lit(pipeline_run_id)).withColumn("batch_id", lit(batch_id)).withColumn("algorithm_name", lit(algorithm_name))
predictions_with_run_id.createOrReplaceTempView(TARGET_TABLE)
print(f"\n✓ Successfully appended new predictions for Run ID: {pipeline_run_id}")
# Optional: Display a sample of the saved data for verification
display(predictions_with_run_id.limit(5))




✓ Successfully appended new predictions for Run ID: 307c3839b41540e7a12f36c8e112b337


customer_id,recency,frequency,monetary,tenure_days,churned,features,scaled_features,rawPrediction,probability,prediction,pipeline_run_id,batch_id,algorithm_name
CUST_000009,130,9,67.51,226,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""130.0"",""9.0"",""67.51"",""226.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.2340966353840512"",""1.8826134912917025"",""0.0706465641047459"",""1.290830360814004""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.8328184269993775"",""0.8328184269993775""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3030494586206253"",""0.6969505413793746""]}",1.0,307c3839b41540e7a12f36c8e112b337,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000012,263,1,46.17,276,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""263.0"",""1.0"",""46.17"",""276.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.49667242389235"",""0.20917927681018916"",""0.04831509205623046"",""1.5764122990471905""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.4796743615084775"",""2.4796743615084775""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07729542375117388"",""0.9227045762488262""]}",1.0,307c3839b41540e7a12f36c8e112b337,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000014,53,3,763.56,341,0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""53.0"",""3.0"",""763.56"",""341.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5031317051950364"",""0.6275378304305674"",""0.7990355575147352"",""1.9476688187503333""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.10360576534868648"",""-0.10360576534868648""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5258782970073336"",""0.4741217029926664""]}",0.0,307c3839b41540e7a12f36c8e112b337,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000022,27,9,192.56,690,0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""27.0"",""9.0"",""192.56"",""690.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.25631237811822605"",""1.8826134912917025"",""0.20150647880328648"",""3.9410307476179764""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.45779723999582994"",""-0.45779723999582994""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6124914903307646"",""0.38750850966923545""]}",0.0,307c3839b41540e7a12f36c8e112b337,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000024,226,1,944.11,184,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""226.0"",""1.0"",""944.11"",""184.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.145429535359966"",""0.20917927681018916"",""0.9879740429111488"",""1.050941532698127""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.03851680283468"",""2.03851680283468""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.11521784689115448"",""0.8847821531088456""]}",1.0,307c3839b41540e7a12f36c8e112b337,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression


In [0]:
%sql
SELECT * FROM churn_predictions_tmp


customer_id,recency,frequency,monetary,tenure_days,churned,features,scaled_features,rawPrediction,probability,prediction,pipeline_run_id,batch_id,algorithm_name
CUST_000009,130,9,67.51,226,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""130.0"",""9.0"",""67.51"",""226.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.2340966353840512"",""1.8826134912917025"",""0.0706465641047459"",""1.290830360814004""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.8328184269993775"",""0.8328184269993775""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3030494586206253"",""0.6969505413793746""]}",1.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000012,263,1,46.17,276,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""263.0"",""1.0"",""46.17"",""276.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.49667242389235"",""0.20917927681018916"",""0.04831509205623046"",""1.5764122990471905""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.4796743615084775"",""2.4796743615084775""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07729542375117388"",""0.9227045762488262""]}",1.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000014,53,3,763.56,341,0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""53.0"",""3.0"",""763.56"",""341.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5031317051950364"",""0.6275378304305674"",""0.7990355575147352"",""1.9476688187503333""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.10360576534868648"",""-0.10360576534868648""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5258782970073336"",""0.4741217029926664""]}",0.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000022,27,9,192.56,690,0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""27.0"",""9.0"",""192.56"",""690.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.25631237811822605"",""1.8826134912917025"",""0.20150647880328648"",""3.9410307476179764""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.45779723999582994"",""-0.45779723999582994""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6124914903307646"",""0.38750850966923545""]}",0.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000024,226,1,944.11,184,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""226.0"",""1.0"",""944.11"",""184.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.145429535359966"",""0.20917927681018916"",""0.9879740429111488"",""1.050941532698127""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.03851680283468"",""2.03851680283468""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.11521784689115448"",""0.8847821531088456""]}",1.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000025,130,7,1398.49,269,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""130.0"",""7.0"",""1398.49"",""269.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.2340966353840512"",""1.464254937671324"",""1.4634648709057339"",""1.5364308276945444""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.8475396002007738"",""0.8475396002007738""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2999492370936806"",""0.7000507629063194""]}",1.0,983d20462fe24d06b1259cb99d93fee9,70a74319-2c68-4c89-970c-272f4b34c90e,logistic_regression
CUST_000026,206,2,246.48,338,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""206.0"",""2.0"",""246.48"",""338.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.9555685145316506"",""0.4183585536203783"

In [0]:
%sql
DESCRIBE churn_predictions_tmp

col_name,data_type,comment
customer_id,string,null
recency,bigint,null
frequency,bigint,null
monetary,double,null
tenure_days,bigint,null
churned,int,null
features,vector,null
scaled_features,vector,null
rawPrediction,vector,null
probability,vector,null


In [0]:
%sql

INSERT INTO churn_predictions_raw (
  customer_id,
  recency,
  frequency,
  monetary,
  tenure_days,
  Churned,
  features_vector,
  scaled_features,  
  probability_vector,
  prediction,  
  pipeline_run_id,
  batch_id,
  algorithm_name  
)
SELECT 
  customer_id,
  recency,
  frequency,
  monetary,
  tenure_days,
  Churned,
  features,
  scaled_features,  
  probability,
  prediction,  
  pipeline_run_id,
  batch_id,
  algorithm_name
FROM churn_predictions_tmp;

num_affected_rows,num_inserted_rows
2050,2050


In [0]:
# ✓ All metrics calculated

# ================================================================================
# MODEL PERFORMANCE METRICS
# ================================================================================
# AUC-ROC:          0.9999
# AUC-PR:           1.0000
# Accuracy:         0.9166
# Precision:        0.8991
# Recall:           1.0000
# F1-Score:         0.9468
# Specificity:      0.6755
# ================================================================================

# Confusion Matrix:
# True Negatives:   356
# False Positives:  171
# False Negatives:  0
# True Positives:   1523
# ================================================================================

In [0]:

# ========================================================================
# Compare Models
# ========================================================================
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

metrics=pipeline_lr.evaluator.get_metrics_summary()


MODEL COMPARISON


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
print(f"Using Run ID: {pipeline_run_id}")
print(f"Using Batch ID: {batch_id}")

# --- 1. Prepare Data for DataFrame Creation ---

# Combine metrics and identifiers into a single dictionary for the row
data_row = {
    'run_id': pipeline_run_id,
    'batch_id': batch_id,
    'algorithm_name': algorithm_name,
    **metrics  # Unpack the metrics dictionary
}

# Convert the single row dictionary into a list of dictionaries for Spark
data = [data_row]

# --- 2. Define Schema (Highly Recommended for Delta Inserts) ---

# Define the exact schema matching the 'model_metrics_summary' Delta table
schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("algorithm_name", StringType(), False),    
    StructField("auc_roc", DoubleType(), True),
    StructField("auc_pr", DoubleType(), True),
    StructField("accuracy", DoubleType(), True),
    StructField("precision", DoubleType(), True),
    StructField("recall", DoubleType(), True),
    StructField("f1_score", DoubleType(), True),
    StructField("specificity", DoubleType(), True),
    StructField("true_positives", LongType(), True),
    StructField("true_negatives", LongType(), True),
    StructField("false_positives", LongType(), True),
    StructField("false_negatives", LongType(), True),
])

# Create the Spark DataFrame with the defined schema
metrics_df = spark.createDataFrame(data, schema=schema)

Using Run ID: 307c3839b41540e7a12f36c8e112b337
Using Batch ID: 70a74319-2c68-4c89-970c-272f4b34c90e


In [0]:
temp_view_name = "current_run_metrics_temp"
metrics_df.createOrReplaceTempView(temp_view_name)
print(f"\nSuccessfully created temporary view: '{temp_view_name}'")
metrics_df.printSchema()



Successfully created temporary view: 'current_run_metrics_temp'
root
 |-- run_id: string (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- algorithm_name: string (nullable = false)
 |-- auc_roc: double (nullable = true)
 |-- auc_pr: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- precision: double (nullable = true)
 |-- recall: double (nullable = true)
 |-- f1_score: double (nullable = true)
 |-- specificity: double (nullable = true)
 |-- true_positives: long (nullable = true)
 |-- true_negatives: long (nullable = true)
 |-- false_positives: long (nullable = true)
 |-- false_negatives: long (nullable = true)



In [0]:
%sql
INSERT INTO gold.model_metrics_summary
(
    run_id,
    batch_id,
    algorithm_name,
    auc_roc,
    auc_pr,
    accuracy,
    precision,
    recall,
    f1_score,
    specificity,
    true_positives,
    true_negatives,
    false_positives,
    false_negatives

)
SELECT 
    run_id,
    batch_id,
    algorithm_name,
    auc_roc,
    auc_pr,
    accuracy,
    precision,
    recall,
    f1_score,
    specificity,
    true_positives,
    true_negatives,
    false_positives,
    false_negatives
FROM current_run_metrics_temp    

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- Create aggregated customer segments
CREATE OR REPLACE TEMPORARY VIEW customer_segments AS
SELECT 
  -- Segment definitions
  CASE 
    WHEN prediction = 1 AND recency > 180 THEN 'Dormant_HighRisk'
    WHEN prediction = 1 AND monetary > 500 THEN 'HighValue_AtRisk' 
    WHEN prediction = 1 AND frequency < 3 THEN 'LowEngagement_Risk'
    WHEN prediction = 1 AND tenure_days < 90 THEN 'NewCustomer_Risk'
    WHEN prediction = 1 THEN 'General_AtRisk'
    WHEN prediction = 0 AND monetary > 1000 THEN 'HighValue_Safe'
    ELSE 'LowRisk_Stable'
  END as segment_name,
  
  -- Aggregated metrics
  COUNT(*) as customer_count,
  ROUND(AVG(recency), 1) as avg_recency_days,
  ROUND(AVG(frequency), 1) as avg_frequency,
  ROUND(AVG(monetary), 2) as avg_monetary_value,
  ROUND(AVG(tenure_days), 1) as avg_tenure_days,
  ROUND(
    AVG(
      CAST(get_json_object(probability_vector, '$[1]') AS DOUBLE)
    ), 3
  ) as avg_churn_probability,
  SUM(monetary) as total_customer_value,
  
  -- Segment characteristics
  COUNT(CASE WHEN recency > 120 THEN 1 END) as count_inactive_120plus,
  COUNT(CASE WHEN monetary > 500 THEN 1 END) as count_high_value,
  COUNT(CASE WHEN frequency < 2 THEN 1 END) as count_low_engagement

FROM churn_predictions_raw
WHERE pipeline_run_id = pipeline_run_id    -- Latest predictions only
GROUP BY 1;

In [0]:
%sql
SELECT * FROM customer_segments

segment_name,customer_count,avg_recency_days,avg_frequency,avg_monetary_value,avg_tenure_days,avg_churn_probability,total_customer_value,count_inactive_120plus,count_high_value,count_low_engagement
General_AtRisk,296,119.5,7.1,137.05,412.3,0.661,40566.670000000006,139,0,0
Dormant_HighRisk,1035,273.4,4.5,451.2,437.0,0.918,466993.94000000024,1035,204,343
LowRisk_Stable,323,30.7,4.6,211.67,427.1,0.405,68368.21000000002,0,35,109
HighValue_AtRisk,136,117.5,4.8,1812.81,409.9,0.66,246542.51999999996,64,136,44
LowEngagement_Risk,227,122.2,1.2,141.67,408.2,0.671,32159.029999999973,117,0,173
HighValue_Safe,33,33.2,5.2,1938.86,449.4,0.416,63982.219999999994,0,33,8


In [0]:
analyst = DeepseekSegmentAnalyst(
    spark,
    secret_scope="deepseek-secrets",
    secret_key_name="API_KEY",
    model="deepseek-chat",                                # or "deepseek-reasoner"
    base_url="https://api.deepseek.com/v1",               # IMPORTANT: DeepSeek endpoint
)


In [0]:
# Optional: sanity check your key + endpoint
analyst.verify_connection()   # should print models or an auth error with detailss

[DeepseekSegmentAnalyst] DeepSeek connection OK. Models: ['deepseek-chat', 'deepseek-reasoner']


In [0]:
segments_df = analyst.build_segments("churn_predictions_raw")
display(segments_df)

segment_name,customer_count,avg_recency_days,avg_frequency,avg_monetary_value,avg_tenure_days,avg_churn_probability,total_customer_value,count_inactive_120plus,count_high_value,count_low_engagement
General_AtRisk,296,119.5,7.1,137.05,412.3,0.661,40566.670000000006,139,0,0
Dormant_HighRisk,1035,273.4,4.5,451.2,437.0,0.918,466993.94000000024,1035,204,343
LowRisk_Stable,323,30.7,4.6,211.67,427.1,0.405,68368.21000000002,0,35,109
HighValue_AtRisk,136,117.5,4.8,1812.81,409.9,0.66,246542.51999999996,64,136,44
LowEngagement_Risk,227,122.2,1.2,141.67,408.2,0.671,32159.029999999973,117,0,173
HighValue_Safe,33,33.2,5.2,1938.86,449.4,0.416,63982.219999999994,0,33,8


In [0]:
#display(segments_df)
# 2) Call DeepSeek for each segment and get a Pandas DataFrame of insights
results_pdf = analyst.analyze_segments(segments_df, throttle_sec=0.2)  # throttle to be nice to rate limits
display(results_pdf)

[DeepseekSegmentAnalyst] Analyzing segment 1/6: General_AtRisk
[DeepseekSegmentAnalyst] Analyzing segment 2/6: Dormant_HighRisk
[DeepseekSegmentAnalyst] Analyzing segment 3/6: LowRisk_Stable
[DeepseekSegmentAnalyst] Analyzing segment 4/6: HighValue_AtRisk
[DeepseekSegmentAnalyst] Analyzing segment 5/6: LowEngagement_Risk
[DeepseekSegmentAnalyst] Analyzing segment 6/6: HighValue_Safe


segment_name,customer_count,avg_recency_days,avg_frequency,avg_monetary_value,avg_tenure_days,avg_churn_probability,total_customer_value,count_inactive_120plus,count_high_value,count_low_engagement,executive_insight
General_AtRisk,296,119.5,7.1,137.05,412.3,0.661,40566.670000000006,139,0,0,"**Risk Assessment:** This segment represents high-volume erosion risk with 296 accounts collectively valued at $40,566 facing 66% probable churn, driven primarily by prolonged inactivity rather than low initial engagement. **Primary Drivers:** The core issue is customer dormancy, with average inactivity of 119.5 days exceeding the 120-day high-risk threshold—nearly half (139 customers) are already deep in this zone, despite solid initial purchase frequency (7.1) and tenure (412 days). **Targeted Actions:** 1) Launch a 30-day ""Reactivation Sprint"" offering tiered discounts based on prior purchase count 2) Deploy personalized win-back outreach highlighting missed product features aligned to their historical purchase categories 3) Implement inactivity-triggered email sequences starting at day 90 to prevent future drift **Priority & Timeline:** Critical priority requiring immediate execution; first campaign deployment within 7 days, with performance review at 30-day mark. **Expected Impact:** Conservatively projecting 25% reactivation (74 customers) would salvage ~$10,140 in at-risk revenue while rebuilding engagement patterns."
Dormant_HighRisk,1035,273.4,4.5,451.2,437.0,0.918,466993.94000000024,1035,204,343,"**Risk Assessment:** This segment represents $467K in immediate revenue at risk, with 91.8% churn probability indicating near-certain departure without intervention. The 204 high-value customers ($500+) within this group amplify the financial exposure. **Primary Drivers:** Extreme dormancy (273 days since last activity) is the core issue, compounded by low purchase frequency (4.5 over 437 days) suggesting failed habit formation or product misalignment. **Targeted Actions:** 1) Deploy a ""Reactivation Campaign"" with high-value incentives (e.g., 30% discount) for high-value dormant accounts. 2) Launch a personalized win-back outreach from senior account managers to the top 20% by historical value. 3) Conduct exit surveys for any inbound responses to capture churn reasons. **Priority & Timeline:** Critical priority. Execute within 2 weeks to preempt quarterly churn reporting cycles. **Expected Impact:** A 5-7% re-engagement rate could salvage $23K-$33K in revenue, with additional learnings from survey data to refine future retention playbooks."
LowRisk_Stable,323,30.7,4.6,211.67,427.1,0.405,68368.21000000002,0,35,109,"**Risk Assessment:** Segment's 40.5% churn probability contradicts its ""LowRisk_Stable"" label, risking $68K in revenue. High inactivity (30.7 days since last activity) and 34% low-engagement members signal early disengagement. **Primary Drivers:** Low purchase frequency (4.6 over 427-day tenure) and 109 customers with <2 purchases indicate lack of habitual use, not dissatisfaction. **Targeted Actions:** 1) Deploy reactivation campaigns for 109 low-engagement customers with personalized, high-value offers. 2) Introduce milestone rewards at 6-month intervals to reinforce loyalty among tenured customers. **Priority & Timeline:** High priority; execute within 30 days before inactivity deepens. **Expected Impact:** Potential 25-30% reduction in churn, preserving ~$20K revenue by converting low-engagement users."
HighValue_AtRisk,136,117.5,4.8,1812.81,409.9,0.66,246542.51999999996,64,136,44,"**Risk Assessment:** High-value segment with 66% churn probability poses immediate $246K revenue threat; 47% are critically inactive (>120 days), accelerating attrition risk. **Primary Drivers:** Engagement decay (117.5 days since last activity) combined with low purchase frequency (4.8 over 409-day tenure) indicates value delivery misalignment. **Targeted Actions:** 1) Executive outreach program with personalized success metrics 2) Reactiva

In [0]:
# 3.1 Generate the report
dbfs_path, public_url = analyst.render_html_report(
    results_pdf,
    title="Customer Churn — Executive Segment Insights",
    company_name="MOMO Smart Analytics",   # <- change to your org
    output_path=f"./smart_analytics/churn_insights_{datetime.now():%Y%m%d_%H%M}.html",
    dark_mode=True
)

[DeepseekSegmentAnalyst] HTML report written to: ./smart_analytics/churn_insights_20251026_0356.html
[DeepseekSegmentAnalyst] Public URL (in workspace): ./smart_analytics/churn_insights_20251026_0356.html


In [0]:
# 3.2 Preview in the notebook (optional)
with open(dbfs_path, "r", encoding="utf-8") as f:
    displayHTML(f.read())

<!doctype html>
 
 
 
 
 Customer Churn — Executive Segment Insights 

 
 
 
 
 Customer Churn — Executive Segment Insights 
 Prepared for MOMO Smart Analytics 
 
 Segments 6 
 Customers Covered 2,050 
 Weighted Avg Churn 74.8% 
 Total Value $918,612.59 
 
 

 
 Segment Insights 
 
 
 
 Segment 
 Customers 
 Avg Recency (d) 
 Avg Freq 
 Avg Value 
 Avg Tenure (d) 
 Avg Churn 
 Total Value 
 
 
 
 
 
 
 
 Dormant_HighRisk
 
 
 1,035 
 273.4 
 4.5 
 $451.20 
 437.0 
 91.8% 
 $466,993.94 
 
 
 
 **Risk Assessment:** This segment represents $467K in immediate revenue at risk, with 91.8% churn probability indicating near-certain departure without intervention. The 204 high-value customers ($500+) within this group amplify the financial exposure.

**Primary Drivers:** Extreme dormancy (273 days since last activity) is the core issue, compounded by low purchase frequency (4.5 over 437 days) suggesting failed habit formation or product misalignment.

**Targeted Actions:** 
1) Deploy a "Reactivation Campaign" with high-value incentives (e.g., 30% discount) for high-value dormant accounts. 
2) Launch a personalized win-back outreach from senior account managers to the top 20% by historical value. 
3) Conduct exit surveys for any inbound responses to capture churn reasons.

**Priority & Timeline:** Critical priority. Execute within 2 weeks to preempt quarterly churn reporting cycles.

**Expected Impact:** A 5-7% re-engagement rate could salvage $23K-$33K in revenue, with additional learnings from survey data to refine future retention playbooks. 
 
 
 
 
 
 
 LowEngagement_Risk
 
 
 227 
 122.2 
 1.2 
 $141.67 
 408.2 
 67.1% 
 $32,159.03 
 
 
 
 **Risk Assessment:** 227 low-engagement customers (67.1% churn probability) represent a $32K revenue risk; 51% are already highly inactive (>120 days), indicating churn is likely imminent.

**Primary Drivers:** Stagnation is the core issue, driven by extreme inactivity (122 avg. days since last activity) and minimal purchase frequency (1.2), failing to build habit or value.

**Targeted Actions:**
1. Launch a 7-day "Re-engagement" email/SMS sequence offering a high-value, one-time discount to spark a second purchase.
2. Deploy a "We Miss You" campaign for the 117 highly inactive leads, featuring a compelling content piece or product guide to re-demonstrate value.
3. For the 173 with <2 purchases, trigger a post-purchase "Next Best Action" campaign immediately after their first buy to shortcut the path to a second.

**Priority & Timeline:** High Priority. Execute campaigns within 2 weeks to intercept before the 6-month inactivity mark.

**Expected Impact:** A conservative 15% salvage rate recovers ~$4,800 in immediate revenue and resets the engagement clock, protecting future lifetime value. 
 
 
 
 
 
 
 General_AtRisk
 
 
 296 
 119.5 
 7.1 
 $137.05 
 412.3 
 66.1% 
 $40,566.67 
 
 
 
 **Risk Assessment:** This segment represents high-volume erosion risk with 296 accounts collectively valued at $40,566 facing 66% probable churn, driven primarily by prolonged inactivity rather than low initial engagement.

**Primary Drivers:** The core issue is customer dormancy, with average inactivity of 119.5 days exceeding the 120-day high-risk threshold—nearly half (139 customers) are already deep in this zone, despite solid initial purchase frequency (7.1) and tenure (412 days).

**Targeted Actions:** 
1) Launch a 30-day "Reactivation Sprint" offering tiered discounts based on prior purchase count 
2) Deploy personalized win-back outreach highlighting missed product features aligned to their historical purchase categories 
3) Implement inactivity-triggered email sequences starting at day 90 to prevent future drift

**Priority & Timeline:** Critical priority requiring immediate execution; first campaign deployment within 7 days, with performance review at 30-day mark.

**Expected Impact:** Conservatively projecting 25% reactivation (74 customers) would salvage ~$10,140 in at-risk revenue while rebuilding en

In [0]:
# 3) (Optional) Save to Delta
# analyst.write_results_delta(results_pdf, target_table="analytics.segment_insights", mode="append")

In [0]:
%sql
DROP TABLE IF EXISTS customer_segments;
DROP TABLE IF EXISTS  churn_predictions_tmp;
DROP TABLE IF EXISTS current_run_metrics_temp;    
